In [2]:
# ==============================
# INSTALL LIBRARIES
# ==============================
!pip install xgboost tensorflow

# ==============================
# UPLOAD DATASET
# ==============================
from google.colab import files
uploaded = files.upload()

# ==============================
# IMPORT LIBRARIES
# ==============================
import pandas as pd
import numpy as np
import pickle

from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image

# ==============================
# LOAD DATA
# ==============================
df = pd.read_csv("data.csv")

# ==============================
# 🔥 CLEAN CONDITION COLUMN (IMPORTANT)
# ==============================
df["condition"] = df["condition"].replace({
    "damaged": "non working"
})

# ==============================
# ENCODING
# ==============================
encoders = {}
for col in ['type', 'category', 'condition']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# ==============================
# TRAIN MODEL
# ==============================
X = df.drop("final_price", axis=1)
y = df["final_price"]

model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1
)

model.fit(X, y)

print("✅ ML Model Trained")

# ==============================
# EVALUATION
# ==============================
y_pred = model.predict(X)

print("📊 R² Score:", r2_score(y, y_pred))
print("📉 MAE:", mean_absolute_error(y, y_pred))

# ==============================
# SAVE MODEL
# ==============================
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(encoders, open("encoders.pkl", "wb"))

# ==============================
# BASE PRICE
# ==============================
BASE_PRICES = {
    "mobile": 15, "laptop": 250, "tablet": 250, "smartwatch": 10,
    "microwave": 150, "mixer": 50, "kettle": 15, "iron": 15, "fan": 50,
    "refrigerator": 500, "washing_machine": 250, "ac": 1000,
    "led_tv": 100, "crt_tv": 50, "monitor": 25,
    "printer": 50, "scanner": 25, "cpu": 250, "ups": 150,
    "cables": 50, "chargers": 50, "remote": 10, "keyboard": 5, "toy": 15
}

# ==============================
# DL MODEL
# ==============================
dl_model = MobileNetV2(weights='imagenet')

# ==============================
# IMAGE CLASSIFICATION
# ==============================
def classify_image(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)

    preds = dl_model.predict(img_array)
    decoded = decode_predictions(preds, top=3)[0]

    print("🔍 DL Predictions:", decoded)

    return decoded[0][1]

# ==============================
# SMART MAPPING
# ==============================
def map_label(label):
    label = label.lower()

    if any(word in label for word in ["modem", "adapter", "plug", "power", "charger", "switch"]):
        return "chargers", "scrap"

    if any(x in label for x in ["laptop", "notebook", "computer", "pc"]):
        return "laptop", "gadget"

    if any(x in label for x in ["phone", "mobile", "cellular"]):
        return "mobile", "gadget"

    if any(x in label for x in ["tv", "television", "monitor", "screen"]):
        return "led_tv", "display"

    if any(x in label for x in ["cpu", "hard", "disk", "processor"]):
        return "cpu", "peripheral"

    if any(x in label for x in ["keyboard", "mouse"]):
        return "keyboard", "scrap"

    if "remote" in label:
        return "remote", "scrap"

    if any(x in label for x in ["microwave", "oven"]):
        return "microwave", "appliance"

    if "fan" in label:
        return "fan", "appliance"

    return "mobile", "gadget"

# ==============================
# 🔥 UPDATED PRICE PREDICTION
# ==============================
def predict_price(data):

    if data["category"] == "scrap":
        return data["weight"] * BASE_PRICES[data["type"]]

    data["base_price"] = BASE_PRICES[data["type"]]

    # 🔧 Safe condition handling
    if data["condition"] not in ["working", "partially working", "non working"]:
        print("⚠️ Invalid condition → defaulting to non working")
        data["condition"] = "non working"

    values = []
    for col in ['type', 'category', 'base_price', 'weight', 'condition', 'age']:
        val = data[col]
        if col in encoders:
            val = encoders[col].transform([val])[0]
        values.append(val)

    # 🔥 ML prediction
    ml_price = float(model.predict([values])[0])

    base_price = BASE_PRICES[data["type"]]
    weight = data["weight"]
    age = data["age"]
    condition = data["condition"]

    # ============================
    # 🔥 RULE CORRECTIONS
    # ============================

    # Limit price
    ml_price = min(ml_price, base_price * weight)

    # Age depreciation
    depreciation = (1 - (age * 0.1))
    ml_price = ml_price * depreciation

    # 🔥 NEW CONDITION LOGIC
    if condition == "working":
        ml_price *= 1.0

    elif condition == "partially working":
        ml_price *= 0.7

    elif condition == "non working":
        ml_price *= 0.5

    final_price = max(ml_price, 1)

    print(f"ML Price: {ml_price:.2f} | Final Price: {final_price:.2f}")

    return final_price

# ==============================
# UPLOAD IMAGE
# ==============================
uploaded_img = files.upload()
img_path = list(uploaded_img.keys())[0]

# ==============================
# FINAL PIPELINE
# ==============================
label = classify_image(img_path)
item_type, category = map_label(label)

print("📷 Detected Item:", item_type)

weight = float(input("Enter weight (kg): "))
condition = input("Enter condition (working/partially working/non working): ")
age = int(input("Enter age (years): "))

data = {
    "type": item_type,
    "category": category,
    "weight": weight,
    "condition": condition,
    "age": age
}

price = predict_price(data)

print("💰 Final Predicted Price:", price)

Saving data.csv to data.csv
✅ ML Model Trained
📊 R² Score: 1.0
📉 MAE: 0.017824385315179825
14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Saving laptop.webp to laptop.webp
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1000ms/step
35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
🔍 DL Predictions: [('n03642806', 'laptop', np.float32(0.5792772)), ('n03832673', 'notebook', np.float32(0.32345852)), ('n04264628', 'space_bar', np.float32(0.03350052))]
📷 Detected Item: laptop
Enter weight (kg): 3
Enter condition (working/partially working/non working): working
Enter age (years): 5
ML Price: 90.02 | Final Price: 90.02
💰 Final Predicted Price: 90.01644134521484


In [3]:
from google.colab import files

files.download("model.pkl")
files.download("encoders.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>